# [P2] Q1 B + D: analysis and realizations

**Owner:** Part 2  
**Temporary part notebook.** At integration it is merged into `Q1_neural_spikes.ipynb` and this file is deleted.


## Before writing any code

Tasks: `t_q1b` (analysis and comparison), `t_q1d` (new realizations, how to tell the patterns apart).

Decisions to read on the team board first (board ids): `b_*` (B), `d_n d_props d_plot d_cls d_dur` (D), and the shared `a_rate`.

Do not write code for a task until its decisions are made. Mark your cells with the part tag so the merge is easy.


## Q1 B, step 1: the statistics functions

Decisions applied: `b_stats` (rate and mean interval, CV, Fano, LV, short-interval fraction, serial correlation, CV2), `b_thr` (a curve over thresholds), `b_fano` (a curve over window sizes) and `b_bin` (100 ms bins).

Each function takes the spike times in seconds and returns a number, or `np.nan` when the number is undefined (too few spikes, division by zero). Nothing here depends on the simulation yet: the functions are tested on inputs whose answer we know in advance (decision `g_verify`).

In [ ]:
import numpy as np
from q1_stats import (isi, mean_rate, mean_isi, cv, lv, cv2, isi_lag1_corr, burst_fraction, burst_curve,
                      window_counts, fano, fano_curve, rate_over_time)

# HE: בדיקה 1: מקרים שאת התשובה שלהם אפשר לחשב ביד.
# HE: פעימות ב-0, 1, 4, 5 שניות => מרווחים 1, 3, 1.
s = np.array([0.0, 1.0, 4.0, 5.0])
assert np.allclose(isi(s), [1, 3, 1])
assert mean_rate(s, T=10) == 0.4                      # HE: 4 פעימות ב-10 שניות
assert np.isclose(mean_isi(s), 5 / 3)

# HE: שני מרווחים בלבד, 1 ו-3: ממוצע 2, סטיית תקן (ddof=1) היא שורש 2, ולכן CV = שורש 2 חלקי 2.
two = np.array([0.0, 1.0, 4.0])
assert np.isclose(cv(two), np.sqrt(2) / 2)
# HE: LV = 3 * ((1-3)/(1+3))^2 = 3 * 0.25 = 0.75, ו-CV2 = 2*|1-3|/(1+3) = 1.
assert np.isclose(lv(two), 0.75)
assert np.isclose(cv2(two), 1.0)

# HE: מרווחים מתחלפים 1,2,1,2,1: כל קצר בא אחרי ארוך, ולכן המתאם הסדרתי הוא בדיוק -1.
alt = np.concatenate([[0.0], np.cumsum([1, 2, 1, 2, 1])])
assert np.isclose(isi_lag1_corr(alt), -1.0)
print("hand-computed cases passed")

In [ ]:
# HE: בדיקה 2: נוירון רגולרי לגמרי (פעימה כל 100 מילי-שניות) הוא "אפס שונות": CV, LV, CV2 ו-Fano שווים 0.
regular = np.arange(0, 10, 0.1)
assert np.isclose(cv(regular), 0, atol=1e-9) and np.isclose(lv(regular), 0, atol=1e-9)
assert np.isclose(cv2(regular), 0, atol=1e-9)
assert np.isclose(fano(regular, T=10, window=0.5), 0, atol=1e-9)   # HE: בכל חלון בדיוק 5 פעימות
assert np.isnan(isi_lag1_corr(regular))                             # HE: אין שונות, אין מתאם: nan
assert burst_fraction(regular, 0.010) == 0                          # HE: אין מרווח קצר מ-10 מילי-שניות

# HE: בדיקה 3: נוירון אקראי (פואסוני) צריך לתת בערך 1 בשלושת המדדים CV, LV ו-Fano. מחוללים אותו כאן עם זרע קבוע.
rng = np.random.default_rng(1)
poisson = np.cumsum(rng.exponential(1 / 10, size=20000))            # HE: קצב 10Hz, כ-2000 שניות
T_long = poisson[-1]
poisson = poisson[:-1]
for name, value in [("CV", cv(poisson)), ("LV", lv(poisson)),
                    ("Fano(500ms)", fano(poisson, T_long, 0.5)), ("CV2", cv2(poisson))]:
    print(f"{name:12s}", round(value, 3))
    assert abs(value - 1) < 0.1 if name != "CV2" else abs(value - 1) < 0.15, name
# HE: במרווחים אקראיים אין זיכרון, לכן המתאם הסדרתי קרוב ל-0.
assert abs(isi_lag1_corr(poisson)) < 0.05
# HE: במרווחים מעריכיים, שיעור המרווחים הקצרים מ-10ms הוא 1-exp(-10Hz*0.01s) = 0.0952.
print("short-interval fraction:", round(burst_fraction(poisson, 0.010), 4), "expected", round(1 - np.exp(-0.1), 4))
assert abs(burst_fraction(poisson, 0.010) - (1 - np.exp(-0.1))) < 0.01
print("regular and random-neuron cases passed")

In [ ]:
# HE: בדיקה 4: מקרי קצה. כל מדד שאי אפשר לחשב מחזיר nan, ולא שגיאה (כלל מההוראות).
empty = np.array([])
one = np.array([0.5])
three = np.array([0.1, 0.2, 0.35])
for f in (mean_isi, cv, lv, cv2, isi_lag1_corr, burst_fraction):
    assert np.isnan(f(empty)) and np.isnan(f(one)), f.__name__
assert np.isnan(cv(one)) and np.isnan(lv(three[:2])) and not np.isnan(lv(three))
assert np.isnan(fano(empty, T=5, window=0.5))                       # HE: ממוצע ספירות 0 => nan
assert np.isnan(fano(three, T=0.4, window=0.5))                     # HE: פחות משני חלונות => nan
assert mean_rate(empty, T=5) == 0

# HE: חלונות: הקלטה של 5 שניות וחלון של 0.5 נותנים בדיוק 10 חלונות, וסכום הספירות שווה למספר הפעימות.
c = window_counts(poisson[poisson < 5], T=5, window=0.5)
assert len(c) == 10 and c.sum() == int((poisson < 5).sum())
# HE: קצב לאורך זמן בחלונות של 100ms: 50 חלונות, וממוצע הקצב שווה לקצב הכולל.
r = rate_over_time(poisson[poisson < 5], T=5, bin_size=0.1)
assert len(r) == 50 and np.isclose(r.mean(), len(poisson[poisson < 5]) / 5)
# HE: העקומות מחזירות ערך לכל סף/חלון, באורך שווה לרשימה שהוזנה.
assert len(burst_curve(poisson, np.linspace(0.002, 0.03, 8))) == 8
assert len(fano_curve(poisson, T_long, [0.05, 0.1, 0.5])) == 3
print("edge cases passed: undefined values return nan, never an exception")

**What step 1 shows**

- The functions agree with answers computed by hand (for example, intervals 1 and 3 give CV = 0.707, LV = 0.75, CV2 = 1).
- A perfectly regular neuron gives CV, LV, CV2 and Fano equal to 0, and an undefined serial correlation (`nan`).
- A random (Poisson) neuron gives CV, LV, CV2 and Fano close to 1 (0.998, 1.001, 1.001, 0.996) and a serial correlation near 0. The short-interval fraction below 10 ms is 0.098, close to the theoretical 0.095.
- Undefined cases (no spikes, one spike, too few windows) return `nan`, never an exception.